# Ensemble final — 9105 k=50 + 9106 k=50 = 100 predicciones

El submit definitivo: promedio de 100 modelos (50 del 9105 + 50 del 9106) usando las mismas 50 semillas en ambos experimentos.

**Por qué esta combinación:**
- 100 modelos → máxima reducción de varianza (sd Private ~0.4-0.5)
- 2 pipelines con HP distintos pero validados → diversidad estructural
- 50 semillas alineadas por par → cada semilla contribuye una predicción del 9105 y una del 9106
- Descarta 9107 (más ruidoso y sin evidencia de mejora en Wilcoxon)

**Standalone.** Solo lee predicciones ya guardadas.

In [ ]:
# --- Setup ---
library(data.table)

# AJUSTAR paths si difieren
path_9105 <- "/content/buckets/b1/exp/WF9105/semillas"
path_9106 <- "/content/buckets/b1/exp/WF9106/semillas"

# Directorio de trabajo para outputs
setwd(dirname(path_9105))  # /content/buckets/b1/exp/
setwd("..")
dir.create("kaggle_final", showWarnings = FALSE)

# Las 50 semillas (mismas en 9105 y 9106)
semillas_k50 <- c(
  # 20 originales
  804043, 653561, 703903, 439693, 665857,
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973,
  # 30 nuevas
  287333, 656119, 694919, 189817, 867463,
  791801, 804317, 680831, 330917, 595951,
  777571, 662339, 202667, 526159, 509389,
  865993, 749471, 398833, 153269, 969637,
  374683, 678481, 799333, 687083, 941207,
  213349, 735659, 872843, 803207, 414913
)

stopifnot(length(semillas_k50) == 50)

# Verifico existencia de todos los archivos ANTES de empezar
n_ok_9105 <- 0
n_ok_9106 <- 0
for (semilla in semillas_k50) {
  if (file.exists(paste0(path_9105, "/prediccion_semilla_", semilla, ".txt"))) n_ok_9105 <- n_ok_9105 + 1
  if (file.exists(paste0(path_9106, "/prediccion_semilla_", semilla, ".txt"))) n_ok_9106 <- n_ok_9106 + 1
}

cat("Archivos encontrados:\n")
cat("  9105: ", n_ok_9105, "/50\n", sep = "")
cat("  9106: ", n_ok_9106, "/50\n", sep = "")

if (n_ok_9105 < 50 || n_ok_9106 < 50) {
  stop("Faltan archivos. Correr primero el escalado a k=50 en el experimento correspondiente.")
}
cat("\nTodo OK. Cargando 100 predicciones.\n")

In [ ]:
# --- Cargar las 100 predicciones ---

primer <- fread(paste0(path_9105, "/prediccion_semilla_", semillas_k50[1], ".txt"))
tb_all <- primer[, list(numero_de_cliente)]

for (semilla in semillas_k50) {
  # 9105
  tb <- fread(paste0(path_9105, "/prediccion_semilla_", semilla, ".txt"))
  tb <- tb[match(tb_all$numero_de_cliente, tb$numero_de_cliente)]
  tb_all[, (paste0("p_9105_", semilla)) := tb$prob]

  # 9106
  tb <- fread(paste0(path_9106, "/prediccion_semilla_", semilla, ".txt"))
  tb <- tb[match(tb_all$numero_de_cliente, tb$numero_de_cliente)]
  tb_all[, (paste0("p_9106_", semilla)) := tb$prob]
}

cols_prob <- grep("^p_", colnames(tb_all), value = TRUE)
stopifnot(length(cols_prob) == 100)

cat("100 predicciones cargadas correctamente.\n")

In [ ]:
# --- Análisis local: comparación con k=20 ensemble A ---

# Ensemble final k=100
tb_all[, ens_final := rowMeans(.SD), .SDcols = cols_prob]

# Ensemble solo 9105 k=50
cols_9105_50 <- grep("^p_9105_", colnames(tb_all), value = TRUE)
tb_all[, ens_9105_50 := rowMeans(.SD), .SDcols = cols_9105_50]

# Ensemble solo 9106 k=50
cols_9106_50 <- grep("^p_9106_", colnames(tb_all), value = TRUE)
tb_all[, ens_9106_50 := rowMeans(.SD), .SDcols = cols_9106_50]

# Correlaciones
cat("=== Correlaciones entre ensembles ===\n")
print(round(cor(tb_all[, .(ens_final, ens_9105_50, ens_9106_50)]), 4))

# Top-2000 coincidencia
get_top <- function(col, k = 2000) {
  tmp <- tb_all[, list(numero_de_cliente, p = get(col))]
  setorder(tmp, -p)
  tmp[1:k, numero_de_cliente]
}

top_final <- get_top("ens_final")
top_9105 <- get_top("ens_9105_50")
top_9106 <- get_top("ens_9106_50")

cat("\n=== Coincidencias top-2000 ===\n")
cat("Ensemble final vs 9105 k=50:", length(intersect(top_final, top_9105)),
    "(", round(length(intersect(top_final, top_9105))/2000, 3), ")\n")
cat("Ensemble final vs 9106 k=50:", length(intersect(top_final, top_9106)),
    "(", round(length(intersect(top_final, top_9106))/2000, 3), ")\n")
cat("9105 k=50 vs 9106 k=50:     ", length(intersect(top_9105, top_9106)),
    "(", round(length(intersect(top_9105, top_9106))/2000, 3), ")\n")

In [ ]:
# --- Submit del ensemble final a Kaggle ---
# 5 cortes centrales para verificar la meseta antes de fijar el submit final

competencia <- "data-mining-junior-2026-a"
cortes <- seq(1800, 2200, by = 100)  # 5 cortes centrales

tb_submit <- tb_all[, list(numero_de_cliente, prob = ens_final)]
setorder(tb_submit, -prob)

for (envios in cortes) {
  tb_submit[, Predicted := 0L]
  tb_submit[1:envios, Predicted := 1L]

  archivo <- paste0("./kaggle_final/KAfinal_9105_9106_k100_", envios, ".csv")
  fwrite(tb_submit[, list(numero_de_cliente, Predicted)],
         file = archivo, sep = ",")

  linea <- paste(
    "kaggle competitions submit",
    "-c", competencia,
    "-f", archivo,
    paste0("-m 'Ensemble final 9105k50+9106k50 (100) envios=", envios, "'")
  )

  cat(format(Sys.time(), "%X"), " - submit final envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nSubmits completados. Elegí el corte 2000 (meseta central) como submit definitivo.\n")